# PKG edge tables: build and monthly refresh

**PNC Treasury Management · DSI · Payment Knowledge Graph**

This notebook builds and refreshes the deduplicated edge tables behind the PKG customer and counterparty metrics, from
the Neo4j staging table `dsihd01p_dsi.neo4j_payments`. The same notebook does the full build and the monthly refresh.
Output goes to `bdahd01p_dlcdi1_cdi_tm` as external parquet tables partitioned by `month` ('YYYY-MM').

**How to run.** Run all cells. A month is built when it is missing, was built with other settings, or its source changed
since it was built; every other month is skipped, so an interrupted run continues where it stopped. Don't run two
copies at the same time.

**What happens to each month**
1. The month's source rows are read once and cached.
2. **Completeness gate.** Rows and the last date of each rail are compared with the previous months. A month that looks
   incomplete stops the run before anything is written.
3. The 5C/6C duplicate pairs are found, reading one extra day before the month.
4. The tables are written to a hidden staging folder (`_staging`).
5. **Exact reconciliation.** Source rows, daily edges, monthly edges and counterparty-month must agree on rows and
   dollars in every category. A mismatch stops the run and nothing is published.
6. **Publish.** Each staged folder is renamed into place with a build marker (`_PKG_BUILD`), and the month is logged in
   `pkg_edge_runlog`. Readers see the previous version of the month until the switch.

Each run also re-checks the source of the last `RECHECK_LAST` built months and rebuilds any whose source changed (late
or restated data).

**Cost.** About 10–15 minutes per month; the full 2024-01 → last month build is about 6–8 hours.

## Tables

| Table | Grain | Columns |
|---|---|---|
| `pkg_edge_daily` | one row per (txn_date, src, dst, category) | txn_date, src_id, src_kind, dst_id, dst_kind, category, cpty_id_src, is_self, amount, n_txn, amt_min, amt_max |
| `pkg_edge_monthly` | one row per (src, dst, category) per month | the same keys without txn_date; amount, n_txn, active_days, first_dt, last_dt, amt_min, amt_max |
| `pkg_dedup_5c6c` | one row per dropped 6C duplicate | trans_id_drop, trans_id_keep, tier, dt_drop, dt_keep, acct, amount, cpty_id_drop, cpty_id_keep |
| `pkg_cpty_month` | counterparty × category × type × name × FI per month | cpty_id, cpty_id_src, category, cpty_type, cpty_name, fi, n_txn, amount |
| `pkg_edge_runlog` | one row per month attempt | build_id, month, status (published / stopped / failed), reason, gate result, source fingerprint, totals, minutes |
| `pkg_edge_src_profile` | source profile per month × category (used by the gate and the re-check) | n_rows, amount, n_days, max_dt, max_load_ts |

**Column semantics.**
- `src_kind` / `dst_kind`: `CUST` = a PNC customer, identified by `mdm_id`; `CPTY` = an external counterparty.
- `amount`: the sum of the absolute transaction amounts, DECIMAL(18,2). Every amount is treated as dollars.
- `amt_min` / `amt_max`: the smallest and largest single payment in the edge. They are equal when every payment had the
  same amount, which the periodicity metrics use to recognise fixed-amount payments.

## Rules and why

| Rule | Reason |
|---|---|
| Debit card (8B/8C) excluded | about 45% of rows but 0.5% of dollars; outside the B2B scope |
| 5C/6C deduplication: drop the 6C twin, keep 5C | 6C (P2P feed) re-records about 21% of its rows from 5C (PRT feed): same account, amount, date and name. 5C carries the counterparty bank; 6C never does. Tier 1 = same day, tier 2 = 6C dated one day later |
| Counterparty id = `unq_cpty_id` for 3B, 3C, 7B, 9B; `unq_cpty_acct_id` for all other categories | for 3B/3C, `unq_cpty_acct_id` is only the originator's 8-digit routing number; 7B and 9B have no account id. `cpty_id_src` = A / B records which was used |
| Customers keyed by current `mdm_id` (not `mdm_id_*_orig`) | a stable entity after MDM merges |
| Self-transfers kept, flagged `is_self` | same `mdm_id` on both sides; up to 34% of internal wires |

## For downstream notebooks

A month is ready when `pkg_edge_runlog` has a `published` row for it.

## Open questions with the data owners
- How `unq_cpty_id` is constructed; what `np_key` and `mdm_id_*_orig` mean.
- How returns and reversals are represented (treated as not occurring for now).
- The source history retention: once old months leave the source, these tables are the only copy.

## Settings

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------
SRC_TABLE = "dsihd01p_dsi.neo4j_payments"
TGT_DB    = "bdahd01p_dlcdi1_cdi_tm"
PREFIX    = "pkg_"

# Which months
RUN_FROM = "2024-01"            # first month of the edge history
RUN_TO   = None                 # None = last complete calendar month
RUN_ONLY = None                 # e.g. ["2026-09"] to build only those months; overrides RUN_FROM / RUN_TO
FORCE    = []                   # months to rebuild even if built and unchanged
RECHECK_LAST       = 2          # re-check the source of the last N built months; rebuild if it changed
MAX_MONTHS_PER_RUN = None       # stop after building N months (split a long build over sessions)

# Completeness gate (§2). Rails: ACH = 1–3, WIRE = 4, RTP = 5–6, CHECK = 7, PCARD = 9
GATE_WINDOW    = 3              # compare with the median of the previous N months
GATE_ROWS_LOW  = 0.25           # STOP when a rail has more than 25% fewer rows than that median
GATE_ROWS_HIGH = 0.25           # warn when it has more than 25% more
GATE_USD       = 0.50           # warn when a rail's dollars move more than 50% either way
GATE_TAIL_DAYS = {"ACH": 4, "WIRE": 4, "RTP": 2, "CHECK": 4, "PCARD": 4}   # STOP when a rail's last date is more
                                                                            # than N days before month end
ACCEPT_MONTHS  = []             # months allowed through a STOP after you checked them

# Edge rules (part of the build signature: changing one rebuilds every month)
EXCLUDE_CODES  = {"8B", "8C"}                                   # debit card
B_ID_CODES     = {"3B", "3C", "7B", "8B", "8C", "9B"}           # counterparty id = unq_cpty_id here
KEEP_CAT, DROP_CAT = "5C.RTP_PRT_CPTY_PAYS", "6C.RTP_P2P_CPTY_PAYS"
EXPECTED_CODES = {"1A1", "1A2", "1B", "1C", "2A", "2B", "2C", "3B", "3C", "4A", "4B", "4C", "5A", "5B", "5C",
                  "6A", "6B", "6C", "7A", "7B", "7C", "8B", "8C", "9B"}   # a new code stops the run
RAIL_OF_DIGIT  = {"1": "ACH", "2": "ACH", "3": "ACH", "4": "WIRE", "5": "RTP", "6": "RTP",
                  "7": "CHECK", "8": "DEBIT_CARD", "9": "PCARD"}
LOAD_TS_COL    = "hdfs_load_ts" # part of the source fingerprint

EDGE_VERSION = "edges_v3"       # bump when the build logic changes (rebuilds every month)
FILES = {"dedup_5c6c": 1, "edge_daily": 8, "edge_monthly": 4, "cpty_month": 4}   # parquet files per month
CODEC = "snappy"
SHUFFLE_PARTITIONS = 400

In [ ]:
# ---- Setup ---------------------------------------------------------------------------------------
# Compatibility shims: Cloudera SPARK3 PySpark is older than the NumPy / pandas in the conda env.
import numpy as np
import pandas as pd
import warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _cat in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_cat)

import calendar, datetime as dt, hashlib, html as _html, json, time, uuid
from contextlib import contextmanager
from decimal import Decimal
from urllib.parse import unquote
from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder.appName("pkg_edges_refresh")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g").config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g").config("spark.executor.memoryOverhead", "2g")
         .config("spark.dynamicAllocation.enabled", "true").config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true").config("spark.sql.adaptive.skewJoin.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .enableHiveSupport().getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.sparkContext.setLogLevel("ERROR")
# The cluster's Atlas lineage hook can't reach its Kafka topic and prints the full query plan on every write.
# It doesn't affect the data; its loggers are switched off so the cell outputs stay readable.
_QUIET = ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
          "org.apache.spark.sql.catalyst.util.package")
try:
    _L = spark._jvm.org.apache.logging.log4j
    for _n in _QUIET:
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    try:
        _L1 = spark._jvm.org.apache.log4j
        for _n in _QUIET:
            _L1.LogManager.getLogger(_n).setLevel(_L1.Level.OFF)
    except Exception as _e:
        print("could not silence the Atlas logs:", str(_e)[:120])
pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 80)

# ---- months
def month_range(a, b):
    y, m = map(int, a.split("-")); y2, m2 = map(int, b.split("-")); out = []
    while (y, m) <= (y2, m2):
        out.append(f"{y:04d}-{m:02d}"); y, m = (y + 1, 1) if m == 12 else (y, m + 1)
    return out
def month_bounds(m):
    y, mo = map(int, m.split("-"))
    return f"{m}-01", f"{m}-{calendar.monthrange(y, mo)[1]:02d}"
def prev_months(m, n):
    y, mo = map(int, m.split("-")); out = []
    for _ in range(n):
        y, mo = (y - 1, 12) if mo == 1 else (y, mo - 1); out.insert(0, f"{y:04d}-{mo:02d}")
    return out
_t = dt.date.today().replace(day=1) - dt.timedelta(days=1)
LAST_COMPLETE = f"{_t.year:04d}-{_t.month:02d}"
now_iso = lambda: dt.datetime.now().isoformat(timespec="seconds")
code_of = lambda c: c.split(".")[0]
rail_of = lambda c: RAIL_OF_DIGIT.get(code_of(c)[:1], "OTHER")

# ---- one consolidated HTML output per cell
def _to_num(pdf):
    pdf = pdf.copy()
    for c in pdf.columns:
        if pdf[c].dtype == object and pdf[c].map(lambda v: isinstance(v, Decimal)).any():
            pdf[c] = pdf[c].map(lambda v: float(v) if isinstance(v, Decimal) else v).astype(float)
    return pdf
def tbl(pdf, pct=(), dec=(), flag=None):
    pdf = _to_num(pdf)
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                   fmt[c] = "{:.1%}"
        elif c in dec:                                 fmt[c] = "{:,.2f}"
        elif pd.api.types.is_bool_dtype(pdf[c]):       continue
        elif pd.api.types.is_integer_dtype(pdf[c]):    fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):      fmt[c] = "{:,.0f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:
        sty = sty.hide(axis="index")
    except AttributeError:
        sty = sty.hide_index()
    if flag and flag in pdf.columns:
        colors = {"published": "#d5efd8", "STOP": "#f8d0d0", "FAILED": "#f8d0d0", "WARN": "#fbe7c0",
                  "ACCEPTED": "#fbe7c0", "OK": "#d5efd8", "True": "#d5efd8", "False": "#f8d0d0"}
        _f = lambda v: f"background-color: {colors.get(str(v), '')}"
        sty = sty.map(_f, subset=[flag]) if hasattr(sty, "map") else sty.applymap(_f, subset=[flag])
    sty = sty.set_table_styles([{"selector": "th, td", "props": "padding: 2px 8px; font-size: 12px;"}])
    return sty.to_html()
def sec(title, body):
    return f"<h4 style='margin:14px 0 4px 0'>{_html.escape(title)}</h4>{body}"
def para(text):
    return f"<p style='margin:4px 0; font-size:13px'>{_html.escape(text)}</p>"
def out(*parts):
    display(HTML("<div>" + "".join(parts) + "</div>"))

# ---- HDFS (Hadoop FileSystem API through the JVM)
HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
exists = lambda p: fs_for(p).exists(JPath(p))
def mkdirs(p):
    fs_for(p).mkdirs(JPath(p))
def delete(p):
    return fs_for(p).delete(JPath(p), True) if exists(p) else False
def rename(a, b):
    assert not exists(b), f"rename target already exists: {b}"
    if not fs_for(a).rename(JPath(a), JPath(b)):
        raise RuntimeError(f"rename failed: {a} -> {b}")
def write_text(p, text):
    o = fs_for(p).create(JPath(p), True); o.write(bytearray(text.encode("utf-8"))); o.close()
def read_text(p):
    if not exists(p): return None
    jio = spark._jvm.java.io
    r = jio.BufferedReader(jio.InputStreamReader(fs_for(p).open(JPath(p)), "UTF-8")); lines = []
    line = r.readLine()
    while line is not None:
        lines.append(line); line = r.readLine()
    r.close()
    return "\n".join(lines)
def size_of(p):
    if not exists(p): return 0, 0
    c = fs_for(p).getContentSummary(JPath(p)); return c.getLength(), c.getFileCount()
GB = 1e9

TIMINGS = []
@contextmanager
def timed(stage, month=None):
    t0 = time.time(); yield
    TIMINGS.append({"month": month, "stage": stage, "seconds": time.time() - t0})

print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__} | last complete month {LAST_COMPLETE}")

## §0 Source and target checks

- **Source.** Every category code must be known (`EXPECTED_CODES`), and every column the build uses must exist. All
  problems are listed together.
- **Target.** Tables are created when missing. When a table's columns differ from the expected ones, its definition is
  re-created and every existing month re-attached; data files are never deleted. Months built with other settings are
  rebuilt in §2.

In [ ]:
# ---- §0 Source and target checks -----------------------------------------------------------------
_parts = [r[0] for r in spark.sql(f"SHOW PARTITIONS {SRC_TABLE}").collect()]          # metastore only, no data read
SRC_CATS, SRC_DATES = set(), set()
for _p in _parts:
    _kv = {k.lower(): unquote(v) for k, v in (x.split("=", 1) for x in _p.split("/"))}
    SRC_CATS.add(_kv["category"]); SRC_DATES.add(_kv["trans_dt"])
KEEP_CATS = sorted(c for c in SRC_CATS if code_of(c) not in EXCLUDE_CODES)
SRC_FIRST_MONTH, SRC_LAST_DATE = min(SRC_DATES)[:7], max(SRC_DATES)

_src_cols = sorted({c.lower() for c in spark.table(SRC_TABLE).columns})
NEED_COLS = ["trans_id", "trans_dt", "trans_amt", "category", "mdm_id_pays", "mdm_id_receives", "pnc_dep_acct_receives",
             "unq_cpty_acct_id", "unq_cpty_id", "cpty_name", "cpty_type", "cpty_fin_entity_name", LOAD_TS_COL]
PROBLEMS = []
_unexpected = sorted(c for c in SRC_CATS if code_of(c) not in EXPECTED_CODES)
if _unexpected:
    PROBLEMS.append({"problem": "new category in the source", "detail": ", ".join(_unexpected),
                     "fix": "add its code to EXPECTED_CODES (and to EXCLUDE_CODES or B_ID_CODES if needed)"})
for c in (KEEP_CAT, DROP_CAT):
    if c not in SRC_CATS:
        PROBLEMS.append({"problem": "5C / 6C category not found", "detail": c, "fix": "check KEEP_CAT / DROP_CAT"})
for c in NEED_COLS:
    if c.lower() not in _src_cols:
        PROBLEMS.append({"problem": "source column missing", "detail": c, "fix": "the source table changed"})
if PROBLEMS:
    out(sec(f"§0 stopped: {len(PROBLEMS)} problem(s)", tbl(pd.DataFrame(PROBLEMS))))
    raise AssertionError(f"§0: {len(PROBLEMS)} problem(s), see the table above")

# Build signature: every setting that changes the content of a month
SIG = hashlib.md5(json.dumps({"version": EDGE_VERSION, "src": SRC_TABLE, "exclude": sorted(EXCLUDE_CODES),
                              "b_id": sorted(B_ID_CODES), "keep": KEEP_CAT, "drop": DROP_CAT},
                             sort_keys=True).encode()).hexdigest()[:12]

DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {TGT_DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
REPL = int(fs_for(DB_LOC).getDefaultReplication(JPath(DB_LOC)))
KEYS = ["dedup_5c6c", "edge_daily", "edge_monthly", "cpty_month"]
LOG_KEYS = ["edge_runlog", "edge_src_profile"]
T = {k: f"{TGT_DB}.{PREFIX}{k}" for k in KEYS + LOG_KEYS}
P = {k: f"{DB_LOC}/{PREFIX}{k}" for k in KEYS + LOG_KEYS}

_ek = [("src_id", "string"), ("src_kind", "string"), ("dst_id", "string"), ("dst_kind", "string"),
       ("category", "string"), ("cpty_id_src", "string"), ("is_self", "boolean")]
SCHEMA = {
    "dedup_5c6c": [("trans_id_drop", "string"), ("trans_id_keep", "string"), ("tier", "int"), ("dt_drop", "date"),
                   ("dt_keep", "date"), ("acct", "string"), ("amount", "decimal(18,2)"),
                   ("cpty_id_drop", "string"), ("cpty_id_keep", "string")],
    "edge_daily": [("txn_date", "date")] + _ek + [("amount", "decimal(18,2)"), ("n_txn", "int"),
                   ("amt_min", "decimal(18,2)"), ("amt_max", "decimal(18,2)")],
    "edge_monthly": _ek + [("amount", "decimal(18,2)"), ("n_txn", "int"), ("active_days", "int"), ("first_dt", "date"),
                   ("last_dt", "date"), ("amt_min", "decimal(18,2)"), ("amt_max", "decimal(18,2)")],
    "cpty_month": [("cpty_id", "string"), ("cpty_id_src", "string"), ("category", "string"), ("cpty_type", "string"),
                   ("cpty_name", "string"), ("fi", "string"), ("n_txn", "bigint"), ("amount", "decimal(22,2)")],
    "edge_runlog": [("build_id", "string"), ("month", "string"), ("status", "string"), ("reason", "string"),
                   ("gate", "string"), ("gate_notes", "string"), ("sig", "string"), ("fingerprint", "string"),
                   ("max_load_ts", "string"), ("src_rows", "bigint"), ("dropped_6c", "bigint"), ("edge_rows", "bigint"),
                   ("amount", "decimal(22,2)"), ("daily_edges", "bigint"), ("monthly_edges", "bigint"),
                   ("minutes", "double"), ("started_at", "string"), ("finished_at", "string")],
    "edge_src_profile": [("month", "string"), ("category", "string"), ("n_rows", "bigint"), ("amount", "decimal(22,2)"),
                   ("n_days", "int"), ("max_dt", "date"), ("max_load_ts", "string"), ("profiled_at", "string"),
                   ("profile_source", "string")],
}
PARTITIONED = {k: k in KEYS for k in SCHEMA}
ddl = lambda key: ", ".join(f"`{c}` {t}" for c, t in SCHEMA[key])
spark_schema = lambda key: ddl(key).replace("`", "")
COLS = {k: [c for c, _ in v] for k, v in SCHEMA.items()}

def table_exists(t):
    db, name = t.split(".")
    return len(spark.sql(f"SHOW TABLES IN {db} LIKE '{name}'").collect()) > 0

def ensure_table(key):
    """Create the table, or re-create its definition when the columns differ (external table: data files stay)."""
    t, part = T[key], " PARTITIONED BY (month STRING)" if PARTITIONED[key] else ""
    create = (f"CREATE EXTERNAL TABLE {t} ({ddl(key)}){part} STORED AS PARQUET LOCATION '{P[key]}' "
              "TBLPROPERTIES ('external.table.purge' = 'false')")
    if table_exists(t):
        have = [(f.name.lower(), f.dataType.simpleString()) for f in spark.table(t).schema
                if not (PARTITIONED[key] and f.name.lower() == "month")]
        if have == SCHEMA[key]:
            return "ok"
        kind = [r[1] for r in spark.sql(f"DESCRIBE TABLE EXTENDED {t}").collect() if r[0] == "Type"]
        assert kind and kind[0].strip().upper() == "EXTERNAL", f"{t} is not an external table; not touching it"
        spark.sql(f"ALTER TABLE {t} SET TBLPROPERTIES ('external.table.purge' = 'false')")   # DROP must keep the files
        spark.sql(f"DROP TABLE {t}"); spark.sql(create)
        if PARTITIONED[key]:
            spark.sql(f"ALTER TABLE {t} RECOVER PARTITIONS")
        return "columns changed: definition re-created"
    mkdirs(P[key]); spark.sql(create)
    if PARTITIONED[key]:
        spark.sql(f"ALTER TABLE {t} RECOVER PARTITIONS")
    return "created"

TABLE_ACTIONS = {k: ensure_table(k) for k in KEYS + LOG_KEYS}

_codes = sorted({code_of(c) for c in SRC_CATS})
out(sec("Source", tbl(pd.DataFrame([{
        "table": SRC_TABLE, "first_date": min(SRC_DATES), "last_date": SRC_LAST_DATE, "partitions": len(_parts),
        "categories_kept": len(KEEP_CATS), "excluded": ", ".join(sorted(c for c in _codes if c in EXCLUDE_CODES)),
        "codes": " ".join(_codes)}]))),
    sec("Target tables", tbl(pd.DataFrame([{"table": T[k], "action": v} for k, v in TABLE_ACTIONS.items()]))),
    para(f"Location {DB_LOC} (replication {REPL}). Build signature {SIG}."))
_S0_OK = True

## §0b Transformations

The PySpark functions every month goes through:
- `source_rows(m)`: the month's rows, trimmed, with the counterparty id chosen per category;
- `dedup_pairs(m)`: the 5C/6C pairs whose 6C row falls in the month;
- `edges(rows)`: source rows after dropping the 6C twins, with src / dst, kinds and the self flag;
- `daily_edges`, `monthly_edges`, `cpty_month`: the published tables.

In [ ]:
# ---- §0b Transformations ---------------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"

def nz(c):
    """String, trimmed, blank -> NULL."""
    s = F.trim(F.col(c).cast("string"))
    return F.when(s == "", F.lit(None)).otherwise(s)

def source_rows(m):
    lo, hi = month_bounds(m)
    s = spark.table(SRC_TABLE).filter(F.col("trans_dt").between(lo, hi) & F.col("category").isin(KEEP_CATS))
    use_b = F.split(nz("category"), r"\.").getItem(0).isin(sorted(B_ID_CODES))
    A, B = nz("unq_cpty_acct_id"), nz("unq_cpty_id")
    return s.select(
        nz("trans_id").alias("trans_id"), F.to_date("trans_dt").alias("txn_date"), nz("category").alias("category"),
        nz("mdm_id_pays").alias("mp"), nz("mdm_id_receives").alias("mr"),
        F.when(use_b, F.coalesce(B, A)).otherwise(F.coalesce(A, B)).alias("cid"),
        F.when(use_b & B.isNotNull(), "B").when(use_b & A.isNotNull(), "A")
         .when(A.isNotNull(), "A").when(B.isNotNull(), "B").alias("cid_src"),
        F.abs(F.col("trans_amt").cast("decimal(18,2)")).alias("amt"),
        nz("cpty_name").alias("cpty_name"), nz("cpty_type").alias("cpty_type"), nz("cpty_fin_entity_name").alias("fi"),
        F.col(LOAD_TS_COL).cast("string").alias("load_ts"))

def profile(rows):
    """Rows, dollars, active days, last date and latest load time per month and category."""
    return (rows.groupBy(F.date_format("txn_date", "yyyy-MM").alias("month"), "category")
                .agg(F.count("*").alias("n_rows"), F.sum("amt").cast("decimal(22,2)").alias("amount"),
                     F.countDistinct("txn_date").cast("int").alias("n_days"), F.max("txn_date").alias("max_dt"),
                     F.max("load_ts").alias("max_load_ts")))

def dedup_pairs(m):
    """5C/6C pairs whose 6C row falls in month m: key (date, receiving account, amount, upper-cased name), one-to-one
    pairing (the k-th 5C row of a key pairs with the k-th 6C row). Tier 1 = same day; tier 2 = the 6C row dated one day
    later. One extra day before the month is read, so tier-2 pairs across the month start are found."""
    lo, hi = month_bounds(m)
    lo_ext = (dt.date.fromisoformat(lo) - dt.timedelta(days=1)).isoformat()
    r = (spark.table(SRC_TABLE)
           .filter(F.col("trans_dt").between(lo_ext, hi) & F.col("category").isin(KEEP_CAT, DROP_CAT))
           .select(nz("trans_id").alias("trans_id"), F.to_date("trans_dt").alias("dt"), nz("category").alias("category"),
                   nz("pnc_dep_acct_receives").alias("acct"), F.abs(F.col("trans_amt").cast("decimal(18,2)")).alias("amt"),
                   F.upper(F.regexp_replace(nz("cpty_name"), r"\s+", " ")).alias("nm"), nz("unq_cpty_acct_id").alias("cid"))
           .filter(F.col("acct").isNotNull() & F.col("nm").isNotNull())
           .withColumn("dt0", F.col("dt")))
    A, B = r.filter(F.col("category") == KEEP_CAT), r.filter(F.col("category") == DROP_CAT)
    GK = ["dt", "acct", "amt", "nm"]
    def ranked(df, s):
        w = Window.partitionBy(*GK).orderBy("trans_id")
        return df.withColumn("rn", F.row_number().over(w)).select(
            *GK, "rn", *[F.col(c).alias(f"{c}_{s}") for c in ("trans_id", "dt0", "cid")])
    t1 = ranked(A, "a").join(ranked(B, "b"), GK + ["rn"]).withColumn("tier", F.lit(1))
    A_left = A.join(t1.select(F.col("trans_id_a").alias("trans_id")), "trans_id", "left_anti")
    B_left = B.join(t1.select(F.col("trans_id_b").alias("trans_id")), "trans_id", "left_anti")
    t2 = (ranked(A_left, "a").join(ranked(B_left.withColumn("dt", F.date_sub("dt", 1)), "b"), GK + ["rn"])
            .withColumn("tier", F.lit(2)))
    return (t1.unionByName(t2).filter(F.col("dt0_b") >= F.lit(lo).cast("date"))
              .select(F.col("trans_id_b").alias("trans_id_drop"), F.col("trans_id_a").alias("trans_id_keep"),
                      F.col("tier").cast("int").alias("tier"), F.col("dt0_b").alias("dt_drop"), F.col("dt0_a").alias("dt_keep"),
                      "acct", F.col("amt").alias("amount"), F.col("cid_b").alias("cpty_id_drop"),
                      F.col("cid_a").alias("cpty_id_keep")))

def edges(rows, dd):
    """Rows after dropping the 6C twins, with src / dst ids and kinds, the id source and the self flag."""
    drop = dd.select(F.col("trans_id_drop").alias("trans_id"))
    kept = (rows.filter(F.col("category") != DROP_CAT)
                .unionByName(rows.filter(F.col("category") == DROP_CAT).join(drop, "trans_id", "left_anti")))
    internal = F.col("mp").isNotNull() & F.col("mr").isNotNull()
    return (kept
        .withColumn("src_kind", F.when(F.col("mp").isNotNull(), "CUST").otherwise("CPTY"))
        .withColumn("src_id",   F.coalesce("mp", "cid"))
        .withColumn("dst_kind", F.when(F.col("mr").isNotNull(), "CUST").otherwise("CPTY"))
        .withColumn("dst_id",   F.coalesce("mr", "cid"))
        .withColumn("cpty_id_src", F.when(internal, F.lit(None).cast("string")).otherwise(F.col("cid_src")))
        .withColumn("is_self", internal & (F.col("mp") == F.col("mr"))))

EDGE_KEYS = ["src_id", "src_kind", "dst_id", "dst_kind", "category", "cpty_id_src", "is_self"]

def daily_edges(e):
    return (e.groupBy("txn_date", *EDGE_KEYS)
             .agg(F.sum("amt").cast("decimal(18,2)").alias("amount"), F.count("*").cast("int").alias("n_txn"),
                  F.min("amt").cast("decimal(18,2)").alias("amt_min"), F.max("amt").cast("decimal(18,2)").alias("amt_max")))

def monthly_edges(d):
    return (d.groupBy(*EDGE_KEYS)
             .agg(F.sum("amount").cast("decimal(18,2)").alias("amount"), F.sum("n_txn").cast("int").alias("n_txn"),
                  F.count("*").cast("int").alias("active_days"), F.min("txn_date").alias("first_dt"),
                  F.max("txn_date").alias("last_dt"), F.min("amt_min").cast("decimal(18,2)").alias("amt_min"),
                  F.max("amt_max").cast("decimal(18,2)").alias("amt_max")))

def cpty_month(e):
    return (e.filter(F.col("cpty_id_src").isNotNull())
             .groupBy(F.col("cid").alias("cpty_id"), "cpty_id_src", "category", "cpty_type", "cpty_name", "fi")
             .agg(F.count("*").alias("n_txn"), F.sum("amt").cast("decimal(22,2)").alias("amount")))

SORT = {"dedup_5c6c": (None, ["trans_id_drop"]), "edge_daily": ("src_id", ["src_id", "dst_id", "txn_date"]),
        "edge_monthly": ("src_id", ["src_id", "dst_id"]), "cpty_month": ("cpty_id", ["cpty_id"])}
print("transformations ready")

## §1 Plan: which months to build

A month is built when:
- it has never been built, or its folder has no build marker (earlier notebook or interrupted run);
- it was built with other settings, or its tables come from different builds;
- its source changed since it was built (re-check of the last `RECHECK_LAST` built months: rows, dollars and latest load
  time per category);
- it is listed in `FORCE`.

The completeness gate compares each month with the previous `GATE_WINDOW` months; months before the first one to build
are profiled here from the source when needed (nothing is built).

In [ ]:
# ---- §1 Plan --------------------------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"
MONTHS = list(RUN_ONLY) if RUN_ONLY else month_range(RUN_FROM, RUN_TO or LAST_COMPLETE)
assert MONTHS == sorted(MONTHS) and len(set(MONTHS)) == len(MONTHS), "RUN_ONLY must be sorted months without repeats"
assert MONTHS[-1] <= LAST_COMPLETE, f"{MONTHS[-1]} is not complete yet (last complete month {LAST_COMPLETE})"
assert MONTHS[0] >= SRC_FIRST_MONTH, f"the source starts in {SRC_FIRST_MONTH}"

def read_marker(k, m):
    txt = read_text(f"{P[k]}/month={m}/_PKG_BUILD")
    return json.loads(txt) if txt else None

def fingerprint(prof):
    """Source fingerprint of one month: rows, dollars and latest load time per category."""
    lines = sorted(f"{r.category}|{int(r.n_rows)}|{Decimal(r.amount):.2f}|{r.max_load_ts}" for r in prof.itertuples())
    ts = [v for v in prof.max_load_ts if v is not None]
    return hashlib.md5("\n".join(lines).encode()).hexdigest()[:16], (max(ts) if ts else None)

def profile_from_source(ms):
    if not ms: return {}
    pdf = None
    for m in ms:                                       # one month at a time: partition pruning stays simple
        p = profile(source_rows(m)).toPandas()
        pdf = p if pdf is None else pd.concat([pdf, p], ignore_index=True)
    return {m: pdf[pdf.month == m].reset_index(drop=True) for m in ms}

def save_profiles(prof_by_month, source):
    rows = [p.assign(profiled_at=now_iso(), profile_source=source) for p in prof_by_month.values() if len(p)]
    if not rows: return
    pdf = pd.concat(rows, ignore_index=True)[COLS["edge_src_profile"]]
    pdf["amount"] = pdf.amount.map(lambda v: None if v is None else Decimal(v))
    (spark.createDataFrame(pdf, schema=spark_schema("edge_src_profile"))
          .write.insertInto(T["edge_src_profile"]))

def stored_profiles():
    w = Window.partitionBy("month", "category").orderBy(F.desc("profiled_at"))
    pdf = (spark.table(T["edge_src_profile"]).withColumn("r", F.row_number().over(w)).filter("r = 1")
                .drop("r", "profiled_at", "profile_source").toPandas())
    return {m: g.reset_index(drop=True) for m, g in pdf.groupby("month")}

PLAN = []
for m in MONTHS:
    mk = {k: read_marker(k, m) for k in KEYS}
    has_dir = any(exists(f"{P[k]}/month={m}") for k in KEYS)
    builds = {v.get("build_id") for v in mk.values() if v}
    row = {"month": m, "action": "build", "reason": "", "build_id": None, "fingerprint": None}
    if m in FORCE:                                           row["reason"] = "forced"
    elif not has_dir:                                        row["reason"] = "new"
    elif any(v is None for v in mk.values()):                row["reason"] = "no build marker (earlier notebook or interrupted)"
    elif len(builds) > 1:                                    row["reason"] = "tables from different builds"
    elif any(v.get("sig") != SIG for v in mk.values()):      row["reason"] = "built with other settings"
    else:
        v = mk[KEYS[0]]
        row.update(action="skip", reason="built", build_id=v["build_id"], fingerprint=v["fingerprint"])
    PLAN.append(row)

_built = [r for r in PLAN if r["action"] == "skip"]
_recheck = _built[-RECHECK_LAST:] if RECHECK_LAST else []
with timed("re-check source"):
    _rc_prof = profile_from_source([r["month"] for r in _recheck])
RECHECK = []
for r in _recheck:
    fp, ts = fingerprint(_rc_prof[r["month"]])
    changed = fp != r["fingerprint"]
    RECHECK.append({"month": r["month"], "built_fingerprint": r["fingerprint"], "source_now": fp,
                    "latest_load_ts": ts, "changed": changed})
    if changed:
        r.update(action="build", reason=f"source changed since build {r['build_id']}")
save_profiles(_rc_prof, "recheck")

TODO = [r["month"] for r in PLAN if r["action"] == "build"]
if MAX_MONTHS_PER_RUN:
    TODO = TODO[:MAX_MONTHS_PER_RUN]

PROFILES = stored_profiles()
PROFILES.update(_rc_prof)
_need = sorted({pm for m in TODO for pm in prev_months(m, GATE_WINDOW)} - set(TODO) - set(PROFILES))
_need = [pm for pm in _need if pm >= SRC_FIRST_MONTH]
with timed("gate history"):
    _hist = profile_from_source(_need)
save_profiles(_hist, "gate_history")
PROFILES.update(_hist)

_plan = pd.DataFrame(PLAN)
_summary = (_plan.assign(n=1).groupby(["action", "reason"], as_index=False)
                 .agg(months=("n", "sum"), first=("month", "min"), last=("month", "max")))
out(sec(f"Plan for {MONTHS[0]} → {MONTHS[-1]} ({len(MONTHS)} months): {len(TODO)} to build in this run", tbl(_summary)),
    sec(f"Source re-check of the last {len(RECHECK)} built months", tbl(pd.DataFrame(RECHECK), flag="changed"))
        if RECHECK else para("No built months to re-check."),
    para(f"Gate history profiled from the source: {', '.join(_need) if _need else 'none needed'}."))

## §2 Build

For each month: read and cache, gate, dedup, stage, reconcile, publish. The table below updates after every month.
- **Gate STOP:** the run stops before writing that month. Check the source; if the month really is complete, add it to
  `ACCEPT_MONTHS` and run again.
- **Reconciliation FAILED:** the run stops and nothing is published for that month; the previous version stays.

In [ ]:
# ---- §2 Build -------------------------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"
for k in KEYS:                                     # leftovers of an interrupted or failed run
    delete(f"{P[k]}/_staging"); delete(f"{P[k]}/_trash")

RAILS = ["ACH", "WIRE", "RTP", "CHECK", "PCARD"]
def by_rail(prof):
    p = prof.assign(rail=prof.category.map(rail_of), amt_f=prof.amount.map(float))
    return p.groupby("rail").agg(n_rows=("n_rows", "sum"), amount=("amt_f", "sum"), max_dt=("max_dt", "max"))

def gate(m, prof):
    cur, last = by_rail(prof), dt.date.fromisoformat(month_bounds(m)[1])
    hist_m = [pm for pm in prev_months(m, GATE_WINDOW) if pm in PROFILES and len(PROFILES[pm])]
    hist = pd.concat([by_rail(PROFILES[pm]) for pm in hist_m]) if hist_m else pd.DataFrame()
    rank = {"OK": 0, "WARN": 1, "STOP": 2}
    rows = []
    for rail in [r for r in RAILS if r in cur.index or (len(hist) and r in hist.index)]:
        h = hist.loc[[rail]] if len(hist) and rail in hist.index else pd.DataFrame()
        med_rows = float(h.n_rows.median()) if len(h) else np.nan
        med_usd = float(h.amount.median()) if len(h) else np.nan
        flags, c = [], None
        if rail not in cur.index:
            flags.append(("STOP", "no rows this month"))
        else:
            c = cur.loc[rail]
            behind = (last - c.max_dt).days
            if behind > GATE_TAIL_DAYS.get(rail, 4):
                flags.append(("STOP", f"last date {c.max_dt} ({behind} days before month end)"))
            if len(h):
                rr = c.n_rows / med_rows - 1
                if rr < -GATE_ROWS_LOW:    flags.append(("STOP", f"rows {rr:+.0%} vs median"))
                elif rr > GATE_ROWS_HIGH:  flags.append(("WARN", f"rows {rr:+.0%} vs median"))
                if med_usd:
                    ru = c.amount / med_usd - 1
                    if abs(ru) > GATE_USD: flags.append(("WARN", f"dollars {ru:+.0%} vs median"))
        rows.append({"month": m, "rail": rail, "n_rows": int(c.n_rows) if c is not None else 0,
                     "rows_vs_median": (c.n_rows / med_rows - 1) if c is not None and len(h) else np.nan,
                     "dollars": c.amount if c is not None else 0.0,
                     "dollars_vs_median": (c.amount / med_usd - 1) if c is not None and len(h) and med_usd else np.nan,
                     "last_date": str(c.max_dt) if c is not None else None,
                     "flag": max([f[0] for f in flags], key=rank.get, default="OK"),
                     "note": "; ".join(f[1] for f in flags)})
    det = pd.DataFrame(rows)
    verdict = max(det.flag, key=rank.get) if len(det) else "STOP"
    notes = "; ".join(f"{r.rail}: {r.note}" for r in det.itertuples() if r.note)
    if not hist_m:
        notes = (notes + "; " if notes else "") + "no earlier months to compare (last-date check only)"
    if verdict == "STOP" and m in ACCEPT_MONTHS:
        verdict = "ACCEPTED"
    return {"month": m, "verdict": verdict, "notes": notes, "history": ", ".join(hist_m), "detail": det}

def stage(df, key, m, build_id):
    """Write one table of month m to its staging folder and read it back."""
    path = f"{P[key]}/_staging/month={m}/{build_id}"
    part, sort = SORT[key]
    df = df.repartition(FILES[key], part) if part else df.repartition(FILES[key])
    df.sortWithinPartitions(*sort).select(*COLS[key]).write.mode("overwrite").option("compression", CODEC).parquet(path)
    return path, spark.read.schema(spark_schema(key)).parquet(path)

def staged_schema_ok(key, path):
    files = [s.getPath().getName() for s in fs_for(path).listStatus(JPath(path))]
    if not [n for n in files if n.endswith(".parquet")]:
        return True
    return [(f.name.lower(), f.dataType.simpleString()) for f in spark.read.parquet(path).schema] == SCHEMA[key]

def _d(v):
    """Exact number for comparisons: Decimal amounts stay Decimal, counts become Decimal integers, missing = 0."""
    if v is None: return Decimal(0)
    if isinstance(v, Decimal): return v
    if isinstance(v, (float, np.floating)):
        return Decimal(0) if np.isnan(v) else Decimal(str(v))
    return Decimal(int(v))

def reconcile(prof, e, dd, d, mo, cm):
    rows = e.groupBy("category").agg(
        F.count("*").alias("n_rows"), F.sum("amt").cast("decimal(22,2)").alias("amount"),
        F.sum(F.col("is_self").cast("long")).alias("self_rows"),
        F.sum(F.col("cpty_id_src").isNotNull().cast("long")).alias("ext_rows"),
        F.sum(F.when(F.col("cpty_id_src").isNotNull(), F.col("amt")).otherwise(F.lit(0))).cast("decimal(22,2)").alias("ext_amount"))
    daily = d.groupBy("category").agg(
        F.sum("n_txn").cast("long").alias("d_txn"), F.sum("amount").cast("decimal(22,2)").alias("d_amount"),
        F.count("*").alias("d_edges"),
        F.sum((F.col("src_id").isNull() | F.col("dst_id").isNull()).cast("long")).alias("null_id_edges"),
        F.sum(((F.col("amt_min") > F.col("amt_max")) | (F.col("amount") < F.col("amt_min") * F.col("n_txn"))
               | (F.col("amount") > F.col("amt_max") * F.col("n_txn"))).cast("long")).alias("d_bad_minmax"))
    monthly = mo.groupBy("category").agg(
        F.sum("n_txn").cast("long").alias("m_txn"), F.sum("amount").cast("decimal(22,2)").alias("m_amount"),
        F.count("*").alias("m_edges"), F.sum("active_days").cast("long").alias("m_active_days"),
        F.sum(((F.col("amt_min") > F.col("amt_max")) | (F.col("first_dt") > F.col("last_dt"))).cast("long")).alias("m_bad"))
    cpty = cm.groupBy("category").agg(F.sum("n_txn").cast("long").alias("c_txn"),
                                      F.sum("amount").cast("decimal(22,2)").alias("c_amount"))
    ddr = dd.agg(F.count("*").alias("n_pairs"), F.sum("amount").cast("decimal(22,2)").alias("amount"),
                 F.countDistinct("trans_id_drop").alias("n_drop"), F.countDistinct("trans_id_keep").alias("n_keep"),
                 F.sum((F.col("tier") == 2).cast("long")).alias("tier2")).toPandas().iloc[0]
    df = prof[["category", "n_rows", "amount"]].rename(columns={"n_rows": "raw_rows", "amount": "raw_amount"})
    for q in (rows, daily, monthly, cpty):
        df = df.merge(q.toPandas(), on="category", how="outer")
    for c in df.columns:
        if c != "category":
            df[c] = df[c].map(_d)
    df["dropped"] = [_d(ddr.n_pairs) if c == DROP_CAT else Decimal(0) for c in df.category]
    df["dropped_amount"] = [_d(ddr.amount) if c == DROP_CAT else Decimal(0) for c in df.category]
    checks = {
        "source − 6C duplicates = rows": (df.raw_rows - df.dropped == df.n_rows) & (df.raw_amount - df.dropped_amount == df.amount),
        "rows = daily edges":            (df.n_rows == df.d_txn) & (df.amount == df.d_amount),
        "daily = monthly edges":         (df.d_txn == df.m_txn) & (df.d_amount == df.m_amount) & (df.d_edges == df.m_active_days),
        "external rows = cpty_month":    (df.ext_rows == df.c_txn) & (df.ext_amount == df.c_amount),
        "min/max consistent":            (df.d_bad_minmax == 0) & (df.m_bad == 0)}
    df["ok"] = np.logical_and.reduce(list(checks.values()))
    summary = {k: bool(v.all()) for k, v in checks.items()}
    summary["dedup pairs one-to-one"] = int(ddr.n_pairs) == int(ddr.n_drop) == int(ddr.n_keep)
    return df, summary, ddr

def log_run(**kw):
    pdf = pd.DataFrame([{c: kw.get(c) for c in COLS["edge_runlog"]}])
    pdf["amount"] = pdf.amount.map(lambda v: None if v is None else Decimal(v))
    spark.createDataFrame(pdf, schema=spark_schema("edge_runlog")).write.insertInto(T["edge_runlog"])

def publish(m, build_id, staged, marker):
    """Rename each staged folder into place (old version to _trash first), with the build marker inside."""
    for k in KEYS:
        write_text(f"{staged[k]}/_PKG_BUILD", json.dumps(dict(marker, table=T[k])))
    for k in KEYS:
        final = f"{P[k]}/month={m}"
        mkdirs(P[k])
        if exists(final):
            mkdirs(f"{P[k]}/_trash"); rename(final, f"{P[k]}/_trash/month={m}.{build_id}")
        rename(staged[k], final)
        spark.sql(f"ALTER TABLE {T[k]} ADD IF NOT EXISTS PARTITION (month='{m}') LOCATION '{final}'")
    for k in KEYS:
        delete(f"{P[k]}/_trash"); delete(f"{P[k]}/_staging"); spark.sql(f"REFRESH TABLE {T[k]}")

FAIL_COLS = ["category", "raw_rows", "dropped", "n_rows", "d_txn", "m_txn", "ext_rows", "c_txn", "amount", "d_amount",
             "m_amount", "ok"]
GATE_LOG, QA_LOG, BUILT, PROGRESS = [], [], [], []
_handle = display(HTML(para(f"{len(TODO)} month(s) to build")), display_id=True)
def show_progress(*extra):
    """One output for the whole cell, updated after every month."""
    head = para(f"{len(BUILT)} of {len(TODO)} month(s) published ({TODO[0]} → {TODO[-1]})" if TODO
                else "Nothing to build in this run.")
    body = tbl(pd.DataFrame(PROGRESS), dec=("minutes",), flag="result") if PROGRESS else ""
    _handle.update(HTML("<div>" + head + body + "".join(extra) + "</div>"))

for m in TODO:
    plan_row = next(r for r in PLAN if r["month"] == m)
    build_id = f"{dt.datetime.now():%Y%m%dT%H%M%S}_{uuid.uuid4().hex[:6]}"
    started, t0, logged = now_iso(), time.time(), False
    g, fp, ts, src = None, None, None, None
    try:
        with timed("read + cache source", m):
            src = source_rows(m).persist()
            prof = profile(src).toPandas()
        with timed("gate", m):
            save_profiles({m: prof}, "build")
            PROFILES[m] = prof
            fp, ts = fingerprint(prof)
            g = gate(m, prof); GATE_LOG.append(g)
        if g["verdict"] == "STOP":
            log_run(build_id=build_id, month=m, status="stopped", reason="completeness gate", gate=g["verdict"],
                    gate_notes=g["notes"], sig=SIG, fingerprint=fp, max_load_ts=ts, src_rows=int(prof.n_rows.sum()),
                    minutes=(time.time() - t0) / 60, started_at=started, finished_at=now_iso())
            logged = True
            PROGRESS.append({"month": m, "result": "STOP", "gate": g["verdict"], "minutes": (time.time() - t0) / 60})
            show_progress(sec(f"{m}: the completeness gate stopped the run",
                              tbl(g["detail"], pct=("rows_vs_median", "dollars_vs_median"), flag="flag")),
                          para("Nothing was written for this month. If the source is complete, add the month to "
                               "ACCEPT_MONTHS and run again."))
            raise AssertionError(f"{m}: completeness gate STOP (see the table above)")

        staged = {}
        with timed("5C/6C pairs", m):
            staged["dedup_5c6c"], dd = stage(dedup_pairs(m), "dedup_5c6c", m, build_id)
        e = edges(src, dd)
        with timed("daily edges", m):
            staged["edge_daily"], d = stage(daily_edges(e), "edge_daily", m, build_id)
        with timed("monthly edges", m):
            staged["edge_monthly"], mo = stage(monthly_edges(d), "edge_monthly", m, build_id)
        with timed("counterparty-month", m):
            staged["cpty_month"], cm = stage(cpty_month(e), "cpty_month", m, build_id)
        with timed("reconciliation", m):
            qa, summary, ddr = reconcile(prof, e, dd, d, mo, cm)
            bad_schema = [k for k in KEYS if not staged_schema_ok(k, staged[k])]
        QA_LOG.append({"month": m, "qa": qa, "summary": summary, "dedup": ddr, "gate": g["verdict"]})
        if not all(summary.values()) or bad_schema:
            log_run(build_id=build_id, month=m, status="failed", reason="reconciliation: " + ", ".join(
                        [k for k, v in summary.items() if not v] + [f"schema {k}" for k in bad_schema]),
                    gate=g["verdict"], gate_notes=g["notes"], sig=SIG, fingerprint=fp, max_load_ts=ts,
                    src_rows=int(prof.n_rows.sum()), minutes=(time.time() - t0) / 60, started_at=started,
                    finished_at=now_iso())
            logged = True
            PROGRESS.append({"month": m, "result": "FAILED", "gate": g["verdict"], "minutes": (time.time() - t0) / 60})
            show_progress(sec(f"{m}: RECONCILIATION FAILED (nothing published)", tbl(pd.DataFrame(
                              [{"check": k, "ok": v} for k, v in summary.items()]
                              + [{"check": f"schema of {k}", "ok": False} for k in bad_schema]), flag="ok")),
                          sec("Categories that failed", tbl(qa[~qa.ok][FAIL_COLS], flag="ok")))
            raise AssertionError(f"{m}: reconciliation failed, nothing published (see the table above)")

        n_rows, amount = int(sum(qa.n_rows)), sum(qa.amount)
        d_edges, m_edges = int(sum(qa.d_edges)), int(sum(qa.m_edges))
        with timed("publish", m):
            publish(m, build_id, staged, {"month": m, "build_id": build_id, "sig": SIG, "fingerprint": fp,
                                          "max_load_ts": ts, "built_at": now_iso(), "edge_rows": n_rows})
        mins = (time.time() - t0) / 60
        log_run(build_id=build_id, month=m, status="published", reason=plan_row["reason"], gate=g["verdict"],
                gate_notes=g["notes"], sig=SIG, fingerprint=fp, max_load_ts=ts, src_rows=int(prof.n_rows.sum()),
                dropped_6c=int(ddr.n_pairs), edge_rows=n_rows, amount=amount, daily_edges=d_edges,
                monthly_edges=m_edges, minutes=mins, started_at=started, finished_at=now_iso())
        logged = True
        BUILT.append(m)
        PROGRESS.append({"month": m, "result": "published", "gate": g["verdict"], "rows": n_rows,
                         "dropped_6c": int(ddr.n_pairs), "daily_edges": d_edges, "monthly_edges": m_edges,
                         "minutes": mins, "build_id": build_id})
        show_progress()
    except Exception as ex:
        if not logged:
            log_run(build_id=build_id, month=m, status="failed", reason=f"{type(ex).__name__}: {str(ex)[:400]}",
                    gate=g["verdict"] if g else None, sig=SIG, fingerprint=fp, max_load_ts=ts,
                    minutes=(time.time() - t0) / 60, started_at=started, finished_at=now_iso())
            PROGRESS.append({"month": m, "result": "FAILED", "minutes": (time.time() - t0) / 60})
            show_progress(para(f"{m}: {type(ex).__name__}: {str(ex)[:300]}"))
        raise
    finally:
        if src is not None:
            src.unpersist()

for k in KEYS + LOG_KEYS:
    spark.sql(f"REFRESH TABLE {T[k]}")
show_progress()

## §3 Reconciliation and QA (months built in this run)

In [ ]:
# ---- §3 Reconciliation and QA ----------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"
if QA_LOG:
    per_m = pd.DataFrame([{
        "month": r["month"], "source_rows": int(sum(r["qa"].raw_rows)), "dropped_6c": int(r["dedup"].n_pairs),
        "tier2_pairs": int(r["dedup"].tier2), "edge_rows": int(sum(r["qa"].n_rows)), "amount": float(sum(r["qa"].amount)),
        "daily_edges": int(sum(r["qa"].d_edges)), "monthly_edges": int(sum(r["qa"].m_edges)),
        "null_id_edges": int(sum(r["qa"].null_id_edges)), "gate": r["gate"],
        "reconciled": all(r["summary"].values())} for r in QA_LOG])
    last = QA_LOG[-1]
    d = last["qa"].copy()
    d.insert(1, "code", d.category.map(code_of)); d.insert(2, "rail", d.category.map(rail_of))
    n = d.n_rows.map(float)
    d["daily_per_row"] = d.d_edges.map(float) / n
    d["monthly_per_row"] = d.m_edges.map(float) / n
    d["self_share"] = d.self_rows.map(float) / n
    cols = ["code", "rail", "raw_rows", "dropped", "n_rows", "amount", "d_edges", "daily_per_row", "m_edges",
            "monthly_per_row", "ext_rows", "null_id_edges", "self_share", "ok"]
    checks = pd.DataFrame([{"check": k, "ok": v} for k, v in last["summary"].items()])
    out(sec("Per month: rows, duplicates, edges, reconciliation", tbl(per_m, flag="reconciled")),
        sec(f"Checks for {last['month']} (every month passed the same checks before publishing)", tbl(checks, flag="ok")),
        sec(f"Per category, {last['month']}", tbl(d[cols].sort_values("code"),
            pct=("daily_per_row", "monthly_per_row", "self_share"), flag="ok")))
else:
    out(para("Nothing was built in this run."))

## §4 Completeness gate (months evaluated in this run)

One row per month, then the rail detail for months with a warning. A warning doesn't stop the build; it points at
months worth a look.

In [ ]:
# ---- §4 Completeness gate ----------------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"
if GATE_LOG:
    s = pd.DataFrame([{"month": g["month"], "verdict": g["verdict"], "compared_with": g["history"] or "—",
                       "notes": g["notes"] or "—"} for g in GATE_LOG])
    parts = [sec("Gate per month", tbl(s, flag="verdict"))]
    flagged = [g["detail"] for g in GATE_LOG if g["verdict"] != "OK"]
    if flagged:
        fd = pd.concat(flagged, ignore_index=True)
        parts.append(sec("Rail detail for flagged months", tbl(fd[fd.flag != "OK"],
                         pct=("rows_vs_median", "dollars_vs_median"), flag="flag")))
    out(*parts)
else:
    out(para("No month was evaluated in this run."))

## §5 Status, storage and time

In [ ]:
# ---- §5 Status, storage and time ---------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"
status = []
for m in MONTHS:
    mk = {k: read_marker(k, m) for k in KEYS}
    ok = all(v and v.get("sig") == SIG for v in mk.values()) and len({v["build_id"] for v in mk.values() if v}) == 1
    v = mk[KEYS[0]] or {}
    status.append({"month": m, "built": ok, "build_id": v.get("build_id"), "built_at": v.get("built_at"),
                   "edge_rows": v.get("edge_rows")})
st = pd.DataFrame(status)
built_m = list(st[st.built].month)
size = []
for k in KEYS:
    _sz = [size_of(f"{P[k]}/month={m}") for m in built_m]
    b = sum(x[0] for x in _sz)
    size.append({"table": T[k], "months": len(built_m), "GB_total": b / GB, "GB_per_month": b / max(len(built_m), 1) / GB,
                 f"GB_on_HDFS_x{REPL}": b * REPL / GB, "files_per_month": sum(x[1] for x in _sz) / max(len(built_m), 1)})
S = pd.DataFrame(size)
S = pd.concat([S, pd.DataFrame([{"table": "TOTAL", "months": len(built_m),
              **{c: S[c].sum() for c in S.columns if c not in ("table", "months")}}])], ignore_index=True)
parts = [sec(f"Months {MONTHS[0]} → {MONTHS[-1]}: {len(built_m)} of {len(MONTHS)} built with the current settings",
             tbl(st, flag="built")),
         sec("Storage", tbl(S, dec=tuple(c for c in S.columns if c.startswith("GB") or c == "files_per_month")))]
try:
    q = fs_for(DB_LOC).getQuotaUsage(JPath(DB_LOC))
    if q.getSpaceQuota() >= 0:
        parts.append(para(f"Space quota at {DB_LOC}: {(q.getSpaceQuota() - q.getSpaceConsumed()) / GB / 1000:,.2f} TB "
                          "remaining (replicated)."))
except Exception as ex:
    parts.append(para(f"Quota lookup skipped: {str(ex)[:100]}"))
if TIMINGS:
    TM = pd.DataFrame(TIMINGS)
    per_stage = (TM[TM.month.notna()].groupby("stage", sort=False).seconds.mean() / 60).reset_index(name="minutes_per_month")
    per_month = TM[TM.month.notna()].groupby("month").seconds.sum() / 60
    remaining = len(MONTHS) - len(built_m)
    if len(per_month):
        parts.append(sec("Time per stage (mean over months built in this run)", tbl(per_stage, dec=("minutes_per_month",))))
        parts.append(para(f"Months built in this run: {len(per_month)}, {per_month.mean():,.1f} min each on average. "
                          f"Months still to build: {remaining}"
                          + (f" (about {remaining * per_month.mean() / 60:,.1f} h)." if remaining else ".")))
out(*parts)

## §6 Impala

The tables are standard external parquet tables. Run the printed statements in Impala: `INVALIDATE METADATA` for
tables created or re-defined in this run, `REFRESH` for the others.

In [ ]:
# ---- §6 Impala -----------------------------------------------------------------------------------------
assert globals().get("_S0_OK"), "§0 did not finish: fix the problem it shows, then run all cells again"
for k in KEYS + LOG_KEYS:
    print(f"INVALIDATE METADATA {T[k]};   -- {TABLE_ACTIONS[k]}" if TABLE_ACTIONS[k] != "ok" else f"REFRESH {T[k]};")